# Depth GINN synthetic pretrain diagnostics

这个 notebook 用来诊断当前 `ginn_depth` 合成预训练数据和真实训练输入之间的分布差异。

重点看四类风险：

- synthetic seismic 输入通道是否和真实 `seis / seis_rms` 同量纲；
- synthetic residual / AI / Vp 是否被参数上限或物理边界大量裁剪；
- mask、taper、loss mask 的覆盖是否导致监督区域和扰动区域不一致；
- 正演波形、dynamic gain、速度模型是否引入明显 domain gap。

运行后会在 notebook 输出日志，同时默认写入 `data/output_ginn_depth_synthetic_pretrain_diagnostics_20260506/synthetic_diagnostics.log`，便于后续复盘。

## 0) Setup

可以先用默认采样规模跑一遍。如果速度慢，把 `N_REAL_TRACES` 和 `N_SYNTHETIC_TRACES` 调小。

In [ ]:
from __future__ import annotations

import json
import logging
import math
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from ginn.data import DYNAMIC_GAIN_LOG_CLIP
from ginn_depth.config import DepthGINNConfig
from ginn_depth.data import build_dataset
from ginn_depth.physics import DepthForwardModel
from ginn_depth.synthetic import SyntheticDepthTraceDataset

SEED = 20260506
N_REAL_TRACES = 512
N_SYNTHETIC_TRACES = 512
N_EXAMPLE_TRACES = 4
BATCH_SIZE_DIAG = 16

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_grad_enabled(False)

run_tag = "20260506"
diag_dir = repo_root / "data" / "output_ginn_depth_synthetic_pretrain_diagnostics_20260506"
diag_dir.mkdir(parents=True, exist_ok=True)
log_path = diag_dir / "synthetic_diagnostics.log"

logger = logging.getLogger("ginn_depth.synthetic_diagnostics")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
file_handler = logging.FileHandler(log_path, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.addHandler(file_handler)

logger.info("repo_root=%s", repo_root)
logger.info("diag_dir=%s", diag_dir)
logger.info("seed=%d, n_real=%d, n_synthetic=%d", SEED, N_REAL_TRACES, N_SYNTHETIC_TRACES)

## 1) Load config and datasets

In [ ]:
config_path = repo_root / "experiments" / "ginn_depth" / "train.yaml"
cfg = DepthGINNConfig.from_yaml(config_path, base_dir=repo_root)

logger.info("config_path=%s", config_path)
logger.info(
    "synthetic_pretrain: enabled=%s epochs=%d traces_per_epoch=%d residual_max_abs=%.4f velocity_mode=%s",
    cfg.synthetic_pretrain_enabled,
    cfg.synthetic_pretrain_epochs,
    cfg.synthetic_traces_per_epoch,
    cfg.synthetic_residual_max_abs,
    cfg.synthetic_velocity_mode,
)
logger.info(
    "synthetic_vp: slope=%s intercept=%s blend_alpha=%.3f smooth_samples=%d",
    cfg.synthetic_vp_ai_slope,
    cfg.synthetic_vp_ai_intercept,
    cfg.synthetic_vp_blend_alpha,
    cfg.synthetic_vp_smooth_samples,
)
logger.info(
    "loss/gain: lambda_residual=%.3f lambda_waveform=%.3f gain_source=%s include_dynamic_gain_input=%s",
    cfg.synthetic_lambda_residual,
    cfg.synthetic_lambda_waveform,
    cfg.gain_source,
    cfg.include_dynamic_gain_input,
)

dataset_bundle = build_dataset(cfg)
train_dataset = dataset_bundle.train_dataset

logger.info("train_dataset traces=%d", len(train_dataset))
logger.info("depth_samples=%d", dataset_bundle.depth_axis_m.size)
logger.info("seis_rms=%.8g", train_dataset.seis_rms)
logger.info("lfm_scale=%.8g", train_dataset.lfm_scale)
logger.info("dynamic_gain_median=%s", train_dataset.dynamic_gain_median)
logger.info("input_channel_names=%s", train_dataset.input_channel_names)

## 2) Build synthetic dataset and forward model

In [ ]:
synthetic_dataset = SyntheticDepthTraceDataset(
    train_dataset,
    num_examples=max(N_SYNTHETIC_TRACES, cfg.batch_size),
    residual_max_abs=cfg.synthetic_residual_max_abs,
    log_ai_highpass_samples=cfg.synthetic_log_ai_highpass_samples,
    thin_bed_min_samples=cfg.synthetic_thin_bed_min_samples,
    thin_bed_max_samples=cfg.synthetic_thin_bed_max_samples,
    ai_min=cfg.ai_min,
    ai_max=cfg.ai_max,
    velocity_mode=cfg.synthetic_velocity_mode,
    vp_ai_slope=cfg.synthetic_vp_ai_slope,
    vp_ai_intercept=cfg.synthetic_vp_ai_intercept,
    vp_blend_alpha=cfg.synthetic_vp_blend_alpha,
    vp_smooth_samples=cfg.synthetic_vp_smooth_samples,
)

forward_model = DepthForwardModel(
    dataset_bundle.wavelet_time_s,
    dataset_bundle.wavelet_amp,
    depth_axis_m=dataset_bundle.depth_axis_m,
    amplitude_threshold=cfg.wavelet_amplitude_threshold,
).cpu()

logger.info("synthetic_dataset length=%d", len(synthetic_dataset))
logger.info(
    "synthetic_dataset velocity_clip=[%.3f, %.3f]",
    getattr(synthetic_dataset, "_vp_clip_min", float("nan")),
    getattr(synthetic_dataset, "_vp_clip_max", float("nan")),
)

## 3) Diagnostic helpers

In [ ]:
def to_numpy(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def flatten_values(values, mask=None):
    arr = to_numpy(values).astype(np.float64, copy=False).reshape(-1)
    if mask is not None:
        mask_arr = to_numpy(mask).astype(bool, copy=False).reshape(-1)
        if mask_arr.size != arr.size:
            raise ValueError(f"mask size mismatch: values={arr.size}, mask={mask_arr.size}")
        arr = arr[mask_arr]
    return arr[np.isfinite(arr)]


def robust_stats(values, mask=None):
    arr = flatten_values(values, mask)
    if arr.size == 0:
        return {
            "n": 0,
            "mean": float("nan"),
            "std": float("nan"),
            "rms": float("nan"),
            "min": float("nan"),
            "p01": float("nan"),
            "p05": float("nan"),
            "p50": float("nan"),
            "p95": float("nan"),
            "p99": float("nan"),
            "max": float("nan"),
            "abs_p95": float("nan"),
            "abs_p99": float("nan"),
            "abs_max": float("nan"),
        }
    return {
        "n": int(arr.size),
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "rms": float(np.sqrt(np.mean(arr * arr))),
        "min": float(np.min(arr)),
        "p01": float(np.percentile(arr, 1)),
        "p05": float(np.percentile(arr, 5)),
        "p50": float(np.percentile(arr, 50)),
        "p95": float(np.percentile(arr, 95)),
        "p99": float(np.percentile(arr, 99)),
        "max": float(np.max(arr)),
        "abs_p95": float(np.percentile(np.abs(arr), 95)),
        "abs_p99": float(np.percentile(np.abs(arr), 99)),
        "abs_max": float(np.max(np.abs(arr))),
    }


def log_stats(name, values, mask=None):
    stats = robust_stats(values, mask)
    logger.info(
        "%s | n=%d mean=%.6g std=%.6g rms=%.6g p01=%.6g p50=%.6g p99=%.6g abs_p99=%.6g min=%.6g max=%.6g",
        name,
        stats["n"],
        stats["mean"],
        stats["std"],
        stats["rms"],
        stats["p01"],
        stats["p50"],
        stats["p99"],
        stats["abs_p99"],
        stats["min"],
        stats["max"],
    )
    return stats


def fraction_true(mask):
    arr = to_numpy(mask).astype(bool, copy=False).reshape(-1)
    return float(arr.mean()) if arr.size else float("nan")


def safe_ratio(a, b):
    if not np.isfinite(a) or not np.isfinite(b) or abs(b) < 1e-12:
        return float("nan")
    return float(a / b)


def sanitize_for_json(value):
    if isinstance(value, dict):
        return {str(k): sanitize_for_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [sanitize_for_json(v) for v in value]
    if isinstance(value, np.ndarray):
        return sanitize_for_json(value.tolist())
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return value if math.isfinite(value) else None
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value


def compose_synthetic_input(seismic_norm, lfm_raw, core_mask, dynamic_gain):
    channels = [seismic_norm.squeeze(1)]
    if cfg.include_lfm_input:
        channels.append((lfm_raw / float(train_dataset.lfm_scale)).squeeze(1))
    if cfg.include_mask_input:
        channels.append(core_mask.float().squeeze(1))
    if cfg.include_dynamic_gain_input:
        if dynamic_gain is None:
            channels.append(torch.zeros_like(seismic_norm.squeeze(1)))
        else:
            median = train_dataset.dynamic_gain_median
            if median is None or median <= 0.0:
                median = 1.0
            gain_input = torch.log(torch.clamp(dynamic_gain, min=1e-6) / float(median)).clamp(
                -DYNAMIC_GAIN_LOG_CLIP,
                DYNAMIC_GAIN_LOG_CLIP,
            )
            channels.append(gain_input.squeeze(1))
    return torch.stack(channels, dim=1).float()


def concat_dict_lists(parts):
    return {key: np.concatenate(value, axis=0) for key, value in parts.items() if value}

## 4) Collect real input diagnostics

In [ ]:
def collect_real_diagnostics(dataset, n_traces):
    rng = np.random.default_rng(SEED)
    n = min(int(n_traces), len(dataset))
    indices = rng.choice(len(dataset), size=n, replace=len(dataset) < n)
    channel_names = dataset.input_channel_names
    channel_index = {name: idx for idx, name in enumerate(channel_names)}
    parts = {
        "input_seismic": [],
        "obs": [],
        "lfm_raw": [],
        "lfm_norm": [],
        "velocity_raw": [],
        "mask": [],
        "loss_mask": [],
        "taper_weight": [],
        "dynamic_gain": [],
        "dynamic_gain_input": [],
    }
    for idx in indices:
        item = dataset[int(idx)]
        x = item["input"].detach().cpu().numpy()
        parts["input_seismic"].append(x[channel_index["seismic"]][np.newaxis])
        parts["obs"].append(item["obs"].detach().cpu().numpy())
        parts["lfm_raw"].append(item["lfm_raw"].detach().cpu().numpy())
        if "ai_lfm" in channel_index:
            parts["lfm_norm"].append(x[channel_index["ai_lfm"]][np.newaxis])
        parts["velocity_raw"].append(item["velocity_raw"].detach().cpu().numpy())
        parts["mask"].append(item["mask"].detach().cpu().numpy())
        parts["loss_mask"].append(item["loss_mask"].detach().cpu().numpy())
        parts["taper_weight"].append(item["taper_weight"].detach().cpu().numpy())
        if "dynamic_gain" in item:
            parts["dynamic_gain"].append(item["dynamic_gain"].detach().cpu().numpy())
        if "dynamic_gain_log_ratio" in channel_index:
            parts["dynamic_gain_input"].append(x[channel_index["dynamic_gain_log_ratio"]][np.newaxis])
    collected = concat_dict_lists(parts)
    logger.info("Collected real diagnostics from %d traces", n)
    return collected


real_diag = collect_real_diagnostics(train_dataset, N_REAL_TRACES)

real_stats = {}
real_stats["input_seismic_all"] = log_stats("REAL input_seismic all", real_diag["input_seismic"])
real_stats["input_seismic_mask"] = log_stats("REAL input_seismic core-mask", real_diag["input_seismic"], real_diag["mask"])
real_stats["lfm_raw_mask"] = log_stats("REAL lfm_raw core-mask", real_diag["lfm_raw"], real_diag["mask"])
real_stats["velocity_raw_mask"] = log_stats("REAL velocity_raw core-mask", real_diag["velocity_raw"], real_diag["mask"])
if "dynamic_gain" in real_diag:
    real_stats["dynamic_gain_mask"] = log_stats("REAL dynamic_gain core-mask", real_diag["dynamic_gain"], real_diag["mask"])
if "dynamic_gain_input" in real_diag:
    real_stats["dynamic_gain_input_mask"] = log_stats("REAL dynamic_gain_input core-mask", real_diag["dynamic_gain_input"], real_diag["mask"])

logger.info("REAL mask_active_fraction=%.6f", fraction_true(real_diag["mask"]))
logger.info("REAL loss_mask_active_fraction=%.6f", fraction_true(real_diag["loss_mask"]))
logger.info("REAL taper_positive_fraction=%.6f", fraction_true(real_diag["taper_weight"] > 0.0))

## 5) Collect synthetic diagnostics

In [ ]:
def collect_synthetic_diagnostics(dataset, n_traces, batch_size):
    parts = {
        "input_seismic": [],
        "target_seismic_raw": [],
        "target_residual": [],
        "target_ai": [],
        "lfm_raw": [],
        "velocity_raw": [],
        "raw_reflectivity": [],
        "mask": [],
        "loss_mask": [],
        "taper_weight": [],
        "dynamic_gain": [],
        "dynamic_gain_input": [],
    }
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)
    collected = 0
    channel_names = train_dataset.input_channel_names
    channel_index = {name: idx for idx, name in enumerate(channel_names)}
    for batch in loader:
        if collected >= n_traces:
            break
        take = min(batch["lfm_raw"].shape[0], n_traces - collected)
        batch = {key: value[:take] if torch.is_tensor(value) else value for key, value in batch.items()}
        dynamic_gain = batch.get("dynamic_gain")
        target_seismic = forward_model(batch["target_ai"], batch["velocity_raw"], gain=dynamic_gain)
        x = compose_synthetic_input(target_seismic, batch["lfm_raw"], batch["mask"], dynamic_gain)

        parts["input_seismic"].append(x[:, channel_index["seismic"], :].detach().cpu().numpy()[:, np.newaxis, :])
        parts["target_seismic_raw"].append(target_seismic.detach().cpu().numpy())
        parts["target_residual"].append(batch["target_residual"].detach().cpu().numpy())
        parts["target_ai"].append(batch["target_ai"].detach().cpu().numpy())
        parts["lfm_raw"].append(batch["lfm_raw"].detach().cpu().numpy())
        parts["velocity_raw"].append(batch["velocity_raw"].detach().cpu().numpy())
        parts["raw_reflectivity"].append(batch["raw_reflectivity"].detach().cpu().numpy())
        parts["mask"].append(batch["mask"].detach().cpu().numpy())
        parts["loss_mask"].append(batch["loss_mask"].detach().cpu().numpy())
        parts["taper_weight"].append(batch["taper_weight"].detach().cpu().numpy())
        if dynamic_gain is not None:
            parts["dynamic_gain"].append(dynamic_gain.detach().cpu().numpy())
        if "dynamic_gain_log_ratio" in channel_index:
            parts["dynamic_gain_input"].append(x[:, channel_index["dynamic_gain_log_ratio"], :].detach().cpu().numpy()[:, np.newaxis, :])
        collected += take
    collected_parts = concat_dict_lists(parts)
    logger.info("Collected synthetic diagnostics from %d traces", collected)
    return collected_parts


synthetic_diag = collect_synthetic_diagnostics(synthetic_dataset, N_SYNTHETIC_TRACES, BATCH_SIZE_DIAG)

synthetic_stats = {}
synthetic_stats["input_seismic_all"] = log_stats("SYNTH input_seismic all", synthetic_diag["input_seismic"])
synthetic_stats["input_seismic_mask"] = log_stats("SYNTH input_seismic core-mask", synthetic_diag["input_seismic"], synthetic_diag["mask"])
synthetic_stats["target_seismic_div_real_rms_mask"] = log_stats(
    "SYNTH target_seismic / real_seis_rms core-mask",
    synthetic_diag["target_seismic_raw"] / float(train_dataset.seis_rms),
    synthetic_diag["mask"],
)
synthetic_stats["target_seismic_raw_mask"] = log_stats("SYNTH target_seismic_raw core-mask", synthetic_diag["target_seismic_raw"], synthetic_diag["mask"])
synthetic_stats["target_residual_mask"] = log_stats("SYNTH target_residual core-mask", synthetic_diag["target_residual"], synthetic_diag["mask"])
synthetic_stats["target_ai_mask"] = log_stats("SYNTH target_ai core-mask", synthetic_diag["target_ai"], synthetic_diag["mask"])
synthetic_stats["lfm_raw_mask"] = log_stats("SYNTH lfm_raw core-mask", synthetic_diag["lfm_raw"], synthetic_diag["mask"])
synthetic_stats["velocity_raw_mask"] = log_stats("SYNTH velocity_raw core-mask", synthetic_diag["velocity_raw"], synthetic_diag["mask"])
synthetic_stats["raw_reflectivity"] = log_stats("SYNTH raw_reflectivity", synthetic_diag["raw_reflectivity"])
if "dynamic_gain" in synthetic_diag:
    synthetic_stats["dynamic_gain_mask"] = log_stats("SYNTH dynamic_gain core-mask", synthetic_diag["dynamic_gain"], synthetic_diag["mask"])
if "dynamic_gain_input" in synthetic_diag:
    synthetic_stats["dynamic_gain_input_mask"] = log_stats("SYNTH dynamic_gain_input core-mask", synthetic_diag["dynamic_gain_input"], synthetic_diag["mask"])

logger.info("SYNTH mask_active_fraction=%.6f", fraction_true(synthetic_diag["mask"]))
logger.info("SYNTH loss_mask_active_fraction=%.6f", fraction_true(synthetic_diag["loss_mask"]))
logger.info("SYNTH taper_positive_fraction=%.6f", fraction_true(synthetic_diag["taper_weight"] > 0.0))

## 6) Risk flags and summary logs

这里的阈值是诊断用的经验阈值，不是训练规则。重点看日志中的 ratio 和 fraction。

In [ ]:
mask = synthetic_diag["mask"].astype(bool)
target_ai = synthetic_diag["target_ai"]
target_residual = synthetic_diag["target_residual"]
velocity_raw = synthetic_diag["velocity_raw"]
raw_reflectivity = synthetic_diag["raw_reflectivity"]

ai_lower_hit = float(((target_ai <= cfg.ai_min * (1.0 + 1e-5)) & mask).sum() / max(mask.sum(), 1))
ai_upper_hit = float(((target_ai >= cfg.ai_max * (1.0 - 1e-5)) & mask).sum() / max(mask.sum(), 1))
residual_near_limit = float(((np.abs(target_residual) >= 0.95 * cfg.synthetic_residual_max_abs) & mask).sum() / max(mask.sum(), 1))
reflectivity_nonzero = float((np.abs(raw_reflectivity) > 1e-7).mean())
vp_clip_min = getattr(synthetic_dataset, "_vp_clip_min", float("nan"))
vp_clip_max = getattr(synthetic_dataset, "_vp_clip_max", float("nan"))
vp_low_hit = float(((velocity_raw <= vp_clip_min * (1.0 + 1e-5)) & mask).sum() / max(mask.sum(), 1)) if np.isfinite(vp_clip_min) else float("nan")
vp_high_hit = float(((velocity_raw >= vp_clip_max * (1.0 - 1e-5)) & mask).sum() / max(mask.sum(), 1)) if np.isfinite(vp_clip_max) else float("nan")

real_seis = real_stats["input_seismic_mask"]
synth_seis = synthetic_stats["input_seismic_mask"]
synth_seis_div_real_rms = synthetic_stats["target_seismic_div_real_rms_mask"]
seis_rms_ratio = safe_ratio(synth_seis["rms"], real_seis["rms"])
seis_abs_p99_ratio = safe_ratio(synth_seis["abs_p99"], real_seis["abs_p99"])
seis_div_rms_ratio = safe_ratio(synth_seis_div_real_rms["rms"], real_seis["rms"])
seis_div_abs_p99_ratio = safe_ratio(synth_seis_div_real_rms["abs_p99"], real_seis["abs_p99"])

real_vp = real_stats["velocity_raw_mask"]
synth_vp = synthetic_stats["velocity_raw_mask"]
vp_rms_ratio = safe_ratio(synth_vp["rms"], real_vp["rms"])
vp_abs_p99_ratio = safe_ratio(synth_vp["abs_p99"], real_vp["abs_p99"])

logger.info("=" * 80)
logger.info("SUMMARY / RISK FLAGS")
logger.info("seismic_input_rms_ratio synth/real=%.6g", seis_rms_ratio)
logger.info("seismic_input_abs_p99_ratio synth/real=%.6g", seis_abs_p99_ratio)
logger.info("seismic_if_divided_by_real_seis_rms_rms_ratio synth/real=%.6g", seis_div_rms_ratio)
logger.info("seismic_if_divided_by_real_seis_rms_abs_p99_ratio synth/real=%.6g", seis_div_abs_p99_ratio)
logger.info("vp_rms_ratio synth/real=%.6g", vp_rms_ratio)
logger.info("vp_abs_p99_ratio synth/real=%.6g", vp_abs_p99_ratio)
logger.info("ai_lower_hit_fraction=%.6g", ai_lower_hit)
logger.info("ai_upper_hit_fraction=%.6g", ai_upper_hit)
logger.info("residual_near_config_limit_fraction=%.6g", residual_near_limit)
logger.info("vp_low_clip_fraction=%.6g", vp_low_hit)
logger.info("vp_high_clip_fraction=%.6g", vp_high_hit)
logger.info("reflectivity_nonzero_fraction=%.6g", reflectivity_nonzero)

if np.isfinite(seis_rms_ratio) and (seis_rms_ratio < 0.5 or seis_rms_ratio > 2.0):
    logger.warning("Synthetic seismic input RMS differs from real by >2x. Check whether target_seismic should be divided by train_dataset.seis_rms.")
if ai_lower_hit + ai_upper_hit > 0.02:
    logger.warning("More than 2%% of synthetic AI samples hit ai_min/ai_max. Residual labels may contain boundary artifacts.")
if residual_near_limit > 0.05:
    logger.warning("More than 5%% of residual samples are near synthetic_residual_max_abs. The residual distribution may be dominated by clipping.")
if np.nanmax([vp_low_hit, vp_high_hit]) > 0.02:
    logger.warning("More than 2%% of synthetic Vp samples hit the estimated velocity clip bounds. Check synthetic_vp_ai_* and clip estimation.")
if reflectivity_nonzero < 0.005:
    logger.warning("Synthetic reflectivity is very sparse. Pretrain may mostly see near-LFM samples.")

summary = {
    "run_tag": run_tag,
    "seed": SEED,
    "n_real_traces": N_REAL_TRACES,
    "n_synthetic_traces": N_SYNTHETIC_TRACES,
    "seismic_input_rms_ratio_synth_over_real": seis_rms_ratio,
    "seismic_input_abs_p99_ratio_synth_over_real": seis_abs_p99_ratio,
    "seismic_if_divided_by_real_seis_rms_rms_ratio_synth_over_real": seis_div_rms_ratio,
    "seismic_if_divided_by_real_seis_rms_abs_p99_ratio_synth_over_real": seis_div_abs_p99_ratio,
    "vp_rms_ratio_synth_over_real": vp_rms_ratio,
    "vp_abs_p99_ratio_synth_over_real": vp_abs_p99_ratio,
    "ai_lower_hit_fraction": ai_lower_hit,
    "ai_upper_hit_fraction": ai_upper_hit,
    "residual_near_config_limit_fraction": residual_near_limit,
    "vp_low_clip_fraction": vp_low_hit,
    "vp_high_clip_fraction": vp_high_hit,
    "reflectivity_nonzero_fraction": reflectivity_nonzero,
    "real_stats": real_stats,
    "synthetic_stats": synthetic_stats,
}

summary_path = diag_dir / "synthetic_diagnostics_summary.json"
summary_path.write_text(json.dumps(sanitize_for_json(summary), ensure_ascii=False, indent=2), encoding="utf-8")
logger.info("summary_json=%s", summary_path)
logger.info("log_path=%s", log_path)

## 7) Distribution plots

In [ ]:
def masked_flat(values, mask=None):
    return flatten_values(values, mask)


fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

axes[0, 0].hist(masked_flat(real_diag["input_seismic"], real_diag["mask"]), bins=100, alpha=0.55, density=True, label="real")
axes[0, 0].hist(masked_flat(synthetic_diag["input_seismic"], synthetic_diag["mask"]), bins=100, alpha=0.55, density=True, label="synthetic")
axes[0, 0].set_title("input seismic, core mask")
axes[0, 0].legend()

axes[0, 1].hist(masked_flat(real_diag["lfm_raw"], real_diag["mask"]), bins=100, alpha=0.55, density=True, label="real LFM")
axes[0, 1].hist(masked_flat(synthetic_diag["target_ai"], synthetic_diag["mask"]), bins=100, alpha=0.55, density=True, label="synthetic target AI")
axes[0, 1].axvline(cfg.ai_min, color="0.25", lw=1, ls="--")
axes[0, 1].axvline(cfg.ai_max, color="0.25", lw=1, ls="--")
axes[0, 1].set_title("AI distribution, core mask")
axes[0, 1].legend()

axes[0, 2].hist(masked_flat(synthetic_diag["target_residual"], synthetic_diag["mask"]), bins=100, color="tab:red", alpha=0.75, density=True)
axes[0, 2].axvline(-cfg.synthetic_residual_max_abs, color="0.25", lw=1, ls="--")
axes[0, 2].axvline(cfg.synthetic_residual_max_abs, color="0.25", lw=1, ls="--")
axes[0, 2].set_title("synthetic target residual")

axes[1, 0].hist(masked_flat(real_diag["velocity_raw"], real_diag["mask"]), bins=100, alpha=0.55, density=True, label="real/base Vp")
axes[1, 0].hist(masked_flat(synthetic_diag["velocity_raw"], synthetic_diag["mask"]), bins=100, alpha=0.55, density=True, label="synthetic Vp")
axes[1, 0].axvline(vp_clip_min, color="0.25", lw=1, ls="--")
axes[1, 0].axvline(vp_clip_max, color="0.25", lw=1, ls="--")
axes[1, 0].set_title("Vp distribution, core mask")
axes[1, 0].legend()

axes[1, 1].hist(masked_flat(synthetic_diag["raw_reflectivity"]), bins=100, color="tab:orange", alpha=0.75, density=True)
axes[1, 1].set_title("synthetic raw reflectivity")

if "dynamic_gain_input" in real_diag and "dynamic_gain_input" in synthetic_diag:
    axes[1, 2].hist(masked_flat(real_diag["dynamic_gain_input"], real_diag["mask"]), bins=100, alpha=0.55, density=True, label="real")
    axes[1, 2].hist(masked_flat(synthetic_diag["dynamic_gain_input"], synthetic_diag["mask"]), bins=100, alpha=0.55, density=True, label="synthetic")
    axes[1, 2].legend()
    axes[1, 2].set_title("dynamic gain input")
else:
    axes[1, 2].axis("off")
    axes[1, 2].set_title("dynamic gain input unavailable")

for ax in axes.ravel():
    ax.grid(alpha=0.2)

plot_path = diag_dir / "distribution_diagnostics.png"
fig.savefig(plot_path, dpi=180)
logger.info("distribution_plot=%s", plot_path)

## 8) Example synthetic traces

In [ ]:
z = dataset_bundle.depth_axis_m
examples = [synthetic_dataset[i] for i in range(N_EXAMPLE_TRACES)]

fig, axes = plt.subplots(N_EXAMPLE_TRACES, 6, figsize=(18, 3.2 * N_EXAMPLE_TRACES), sharey=True, constrained_layout=True)
if N_EXAMPLE_TRACES == 1:
    axes = axes[np.newaxis, :]

for row, sample in enumerate(examples):
    lfm_ai = sample["lfm_raw"].squeeze().numpy()
    target_ai = sample["target_ai"].squeeze().numpy()
    residual = sample["target_residual"].squeeze().numpy()
    reflectivity = sample["raw_reflectivity"].squeeze().numpy()
    synthetic_vp = sample["velocity_raw"].squeeze().numpy()
    mask_1d = sample["mask"].squeeze().numpy().astype(bool)
    taper = sample["taper_weight"].squeeze().numpy()
    dynamic_gain = sample.get("dynamic_gain")
    with torch.no_grad():
        seismic = forward_model(
            sample["target_ai"].unsqueeze(0),
            sample["velocity_raw"].unsqueeze(0),
            gain=dynamic_gain.unsqueeze(0) if dynamic_gain is not None else None,
        ).squeeze().numpy()

    axes[row, 0].plot(lfm_ai, z, label="LFM")
    axes[row, 0].plot(target_ai, z, lw=1.1, label="target")
    axes[row, 0].set_title("AI")
    axes[row, 0].legend(fontsize=8)

    axes[row, 1].plot(residual, z, color="tab:red")
    axes[row, 1].axvline(0, color="0.25", lw=0.8)
    axes[row, 1].set_title("logAI residual")

    axes[row, 2].step(reflectivity, z[:-1], where="post", color="tab:orange")
    axes[row, 2].axvline(0, color="0.25", lw=0.8)
    axes[row, 2].set_title("reflectivity")

    axes[row, 3].plot(synthetic_vp, z, color="tab:green")
    axes[row, 3].set_title("synthetic Vp")

    axes[row, 4].plot(seismic, z, color="tab:purple")
    axes[row, 4].set_title("synthetic seismic")

    axes[row, 5].plot(mask_1d.astype(float), z, label="mask")
    axes[row, 5].plot(taper, z, label="taper")
    axes[row, 5].set_title("mask / taper")
    axes[row, 5].legend(fontsize=8)

for ax in axes.ravel():
    ax.invert_yaxis()
    ax.grid(alpha=0.2)

example_plot_path = diag_dir / "example_synthetic_traces.png"
fig.savefig(example_plot_path, dpi=180)
logger.info("example_trace_plot=%s", example_plot_path)

## 9) Quick interpretation checklist

跑完后优先看这些日志：

- `seismic_input_rms_ratio synth/real`：如果明显不是 1 附近，优先检查 synthetic seismic 是否缺少 `seis_rms` 归一化。
- `ai_lower_hit_fraction + ai_upper_hit_fraction`：如果超过几个百分点，说明 AI 边界裁剪可能影响标签。
- `residual_near_config_limit_fraction`：如果偏高，说明 residual 分布被 `synthetic_residual_max_abs` 主导。
- `vp_low_clip_fraction / vp_high_clip_fraction`：如果偏高，说明 AI-Vp 线性关系或 Vp clip 可能过强。
- 分布图里真实和合成的 seismic、Vp、dynamic gain 输入是否明显错位。

把日志或 summary JSON 发回来，就可以继续判断应该改归一化、扰动策略，还是调合成参数。